# IOAI — 2026 Competition Lever Decoding (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    os.system('wget -q https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-competition-lever-decoding/train.csv.gz')
    os.system('gunzip -f train.csv.gz && mv train.csv data/train.csv')
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# JOAI 2026 — Lever Decoding 모범답안 (GBM + temporal neural features)

> **Japan Olympiad in AI 2026 — Playground Competition**

Two improvements over the linear baseline:
1. **HistGradientBoostingRegressor** — captures non-linear neural→lever relationships.
2. **Time-lagged neural features** — within each trial, add the previous timestep's brain signals (`*_lag1`); behaviour decoding benefits from recent neural context.

Group held-out MSE drops from ~1.34 (linear) to ~1.19.

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv('data/train.csv').sort_values('sample_id').reset_index(drop=True)
# GROUP held-out: every 5th sample_id (whole trials held out, no timestep leakage)
sids = sorted(df['sample_id'].unique()); val_set = set(sids[::5])
is_val = df['sample_id'].isin(val_set)
tr, va = df[~is_val], df[is_val]
BRAIN = [c for c in df.columns if c not in ['id','sample_id','mouse_id','lever','day_n','time'] and df[c].dtype != 'O']
print('train', len(tr), 'val', len(va), '| brain-region features:', len(BRAIN))

In [ ]:
# add per-trial lag-1 of every brain signal (decoding context); df already sorted by sample_id
for c in BRAIN:
    df[c+'_lag1'] = df.groupby('sample_id')[c].shift(1)
LAG = [c+'_lag1' for c in BRAIN]
df[LAG] = df[LAG].fillna(0)
is_val = df['sample_id'].isin(val_set)
tr, va = df[~is_val], df[is_val]
FEATS = BRAIN + LAG
print('features:', len(FEATS))

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
m = HistGradientBoostingRegressor(random_state=0, max_iter=300, learning_rate=0.05).fit(tr[FEATS], tr['lever'])
pred = m.predict(va[FEATS])
print('held-out MSE:', round(mean_squared_error(va['lever'], pred), 4))
pd.DataFrame({'id': va['id'].to_numpy(), 'lever': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(va))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)